# Polars I

In this notebook, we will work with the following:

- Reading common research data with pandas and Polars.
- Inspecting a dataframe.
- Cleaning types and names.
- Selecting, filtering, and transforming rows and columns.
- Writing data for later use.


In [ ]:
# imports
from pathlib import Path

import pandas as pd
import polars as pl

In [ ]:
# path
DATA_DIR = Path("..") / "data"

**Aside:** We're using `pathlib` here, which is a way to specify paths to files or directories that's more *Pythonic* and works better cross platform.
It doesn't actually matter here, because we're in a devcontainer that's always running Linux, but it's worth knowing that there's a purpose-built way to do this instead of just using strings.

## Reading data

Polars reads common tabular formats, including CSV, Excel, JSON, and Parquet.
CSV is useful because almost everything can read it, and Parquet is useful because it is fast, compact, and preserves data types nicely.
You can see the broader set of options in the [Polars IO documentation](https://docs.pola.rs/user-guide/io/).

However, we often work with other formats that pandas supports well, including Stata files.
When pandas is the easiest way to read a file, we can read it with pandas and immediately convert it to Polars with [`pl.from_pandas()`](https://docs.pola.rs/api/python/stable/reference/api/polars.from_pandas.html).

In [ ]:
firmyear_pd = pd.read_stata(DATA_DIR / "firmyear.dta")
firmyear = pl.from_pandas(firmyear_pd)

firmyear

## Inspecting data

Often, we start working with data by seeing what exactly we have.
That often includes the size, column names, data types, and some basic descriptive information.

In [ ]:
firmyear.head()

In [ ]:
print(f"Rows, columns: {firmyear.shape}")
print(f"Column names:  {firmyear.columns}")

In [ ]:
firmyear.schema

In [ ]:
firmyear.describe()

In [ ]:
firmyear.null_count()

## Dataframes and expressions

The main object we are using is a Polars [`DataFrame`](https://docs.pola.rs/api/python/stable/reference/dataframe/index.html). A dataframe is the 2D tabular data with rows and columns that we all are likely familiar with.

The other object we will see often is a Polars expression, often written with `pl.col()`. These expressions allow us to express our computations with Polars objects that allow Polars to stay within its highly optimized library code (written in the Rust programming language).

The [Polars expressions documentation](https://docs.pola.rs/user-guide/expressions/) has many more examples.

In [ ]:
pl.col("year")

## Cleaning data

Our Stata data file gives us a few columns as strings. As we often do, we need to change to numeric types for the columns that have actual numeric data that has been stored as strings.

Polars makes transformations using [`with_columns()`](https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.with_columns.html). Inside it, we provide one or more expressions.

For more on changing types, see the [casting documentation](https://docs.pola.rs/user-guide/expressions/casting/).

In [ ]:
firmyear = firmyear.with_columns(
    pl.col("year").cast(pl.Int64),
    pl.col("count_of_employees").cast(pl.Int64),
)

firmyear.schema

For our convenience, we often want to change column names to something that is either clearer or that better captures what our measure represents. Here, we'll change `count_of_employees` to `size_emp` so it is easier to read and indicates that we're using it as a measure of firm size.

In [ ]:
firmyear = firmyear.rename({"count_of_employees": "size_emp"})

firmyear.head()

## Viewing and selecting data

Some datasets come with many columns, and we also sometimes overgather data to prevent needing to go back for more variables later. The `select()` method lets us choose columns and create quick one-off views of the data, often to focus in on the ones relevant to the task at hand.

Using `select()` also has us specify the order we want variables returned in, so we can use it for reordering columns, too.

In [ ]:
firmyear.select("name", "year", "size_emp")

In [ ]:
firmyear.select(
    "name",
    "year",
    (pl.col("size_emp") * 1000).alias("size_emp"),
)

As we see above, `select()` also allows us to transform data using expressions. In this example, we're taking our employee data expressed in thousands and transforming it such that the units are now simply employees (apparently rounded to the nearest thousand).

However, this only changes the result displayed by `select()`; it does not change `firmyear` unless we assign the result.

## Filtering rows

`filter()` keeps rows where a condition (itself an expression) is `True`.

In [ ]:
firmyear.filter(pl.col("name") == "Microsoft")

In [ ]:
firmyear.filter((pl.col("name") == "Microsoft") | (pl.col("year") >= 2018))

## Transforming data

We can also transform data by computing new variables using the data we have combined with operations that we specify. So far, this is still the same firm-year level of the data: one row per firm per year.

We can use `.alias()`, which provides a name for the new column created by an expression.

Some row-level transformations depend on groups. For example, employee growth only makes sense within a firm.

In Polars, `.over("name")` tells the expression to do the calculation separately for each firm. This is an example of a [window expression](https://docs.pola.rs/user-guide/expressions/window-functions/).

Note that we'd prefer using an identifier like CUSIP, GVKEY, or the like instead of the firm name, but that's what we have in this example data.

In [ ]:
firmyear = firmyear.sort("name", "year").with_columns(
    pl.col("size_emp").diff().over("name").alias("size_emp_change"),
    pl.col("size_emp").pct_change().over("name").alias("size_emp_growth"),
)

firmyear

Those first rows for each firm have missing values because there is no prior year within the firm to compare against. We can filter on missingness with `is_null()` and `is_not_null()`. The [missing data documentation](https://docs.pola.rs/user-guide/expressions/missing-data/) shows additional tools for working with null values.

In [ ]:
firmyear.filter(pl.col("size_emp_change").is_not_null())

In [ ]:
firmyear.filter(pl.col("size_emp_change").is_null())

## Saving and exporting

Now that we have a cleaned firm-year dataset, we'll write it in two formats.

- CSV is easy to share and inspect.
- Parquet is usually better for reusing data in Python because it preserves column types and is efficient. R also has tools for reading it.

The [Polars IO documentation](https://docs.pola.rs/user-guide/io/) includes more detail on reading and writing these formats.

In [ ]:
firmyear.write_csv(DATA_DIR / "2026-06-02_firmyear_polars.csv")
firmyear.write_parquet(DATA_DIR / "2026-06-02_firmyear_polars.parquet")

In [ ]:
pl.read_parquet(DATA_DIR / "2026-06-02_firmyear_polars.parquet").head()

## On your own

Try the same basic workflow on the firm-year data again, or on a small dataset of your own.

1. Read the dataset into a Polars dataframe named `my_data`.
1. Display the first 10 rows.
1. Display the schema and notice whether any types look wrong.
1. Select two or three columns.
1. Filter to rows that satisfy one condition.
1. Create one new column with `with_columns()`.
1. Write the result to `../data/` with `_polars` at the end of the filename.

In [ ]:
# 1-1 code

In [ ]:
# 1-2 code

In [ ]:
# 1-3 code

## Bonus content

Many of the ideas in working with data are common across many environments, because the problem space is fundamentally the same. For example, SQL uses the same `OVER` idea for window functions.

We've seen the alias idea, from how we import Python packages, to naming computed results or renaming columns. It also has a SQL equivalent, `AS`.

In general, Polars uses a lot of naming that is similar to SQL naming for the same operations.

Because these share so much conceptual overlap, once you really learn how to think through working with data, you'll often find that you simply need to learn the syntax to express the operations that you already know you would like to use.